# is-differentiable-flag — ex1: three-gate requires_grad reading is_differentiable from closure

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `is-differentiable-flag`. Running the final beacon cell reports progress against the `Backprop: is_differentiable flag` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: is_differentiable flag` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`is-differentiable-flag`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "is-differentiable-flag"
DD_SUBTOPIC = "Backprop: is_differentiable flag"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## is_differentiable flag — quick refresher

`is_differentiable` is a **per-op** static flag, supplied at register time, NOT inferred from inputs or runtime state. It's the second gate in the three-gate `requires_grad` rule:

```
requires_grad = (
    grad_tracking_enabled        # global toggle (no_grad context)
    and is_differentiable        # PER-OP flag, captured in closure
    and any(input.requires_grad for input in tensor_inputs)
)
```

Differences from the global toggle:
- Global toggle changes at RUNTIME (per `with no_grad():`). Per-op flag is set ONCE, at wrap-time.
- Global toggle gates ALL ops. Per-op flag gates only this one op.

Captured via closure: `wrap_forward_fn(fn, is_differentiable=False)` returns a `tensor_func` whose closure remembers the False, so EVERY subsequent call short-circuits to `requires_grad=False` for the output.

### Exercise 1 — three-gate requires_grad reading is_differentiable from closure

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the three-gate requires_grad rule with the is_differentiable flag closed over at wrap-time, distinguishing per-op behavior from the global runtime toggle.
> Keywords: is-differentiable, three-gate, closure, per-op
> ```

**KCs targeted:** `is-differentiable-flag`, `requires-grad-propagation`

Implement TWO related pieces:

**1. `make_check_requires_grad(is_differentiable)`** — a *factory* that captures the per-op `is_differentiable` flag in closure and returns a function `check(args) -> bool`. `check` reads `grad_tracking_enabled` from the module globals each call, ANDs all three gates, and returns the result.

Pseudocode:

```python
def make_check_requires_grad(is_differentiable):
    def check(args):
        return (
            globals()['grad_tracking_enabled']
            and is_differentiable
            and any(
                isinstance(a, MiniTensor) and a.requires_grad for a in args
            )
        )
    return check
```

**2. `set_grad_tracking(enabled: bool)`** — a tiny helper that writes `grad_tracking_enabled` in the module globals. Use `globals()['grad_tracking_enabled'] = enabled`.

**The point of the closure.** `is_differentiable` is set ONCE per op (at wrap-time, via the factory). After that it's fixed — you cannot change it at call-site. By contrast, `grad_tracking_enabled` is read FRESH each call, so flipping the global instantly affects every subsequent `check`.

Test exhaustively covers:
- per-op flag stays sticky across calls.
- runtime toggle flips behavior on/off mid-program.
- mixed factory invocations don't cross-contaminate.
- gate 3 (any input tracked) is unchanged.

Do NOT call `torch.autograd`.

In [ ]:
def make_check_requires_grad(is_differentiable: bool):
    # Closure captures is_differentiable — sticky for the lifetime of `check`.
    def check(args):
        return (
            globals()['grad_tracking_enabled']    # gate 1: runtime
            and is_differentiable                  # gate 2: per-op (closure)
            and any(                               # gate 3: any tracked input
                isinstance(a, MiniTensor) and a.requires_grad
                for a in args
            )
        )
    return check


def set_grad_tracking(enabled: bool):
    globals()['grad_tracking_enabled'] = enabled


<details><summary>Solution</summary>

```python
def make_check_requires_grad(is_differentiable: bool):
    # Closure captures is_differentiable — sticky for the lifetime of `check`.
    def check(args):
        return (
            globals()['grad_tracking_enabled']    # gate 1: runtime
            and is_differentiable                  # gate 2: per-op (closure)
            and any(                               # gate 3: any tracked input
                isinstance(a, MiniTensor) and a.requires_grad
                for a in args
            )
        )
    return check


def set_grad_tracking(enabled: bool):
    globals()['grad_tracking_enabled'] = enabled
```

**Why a factory.** The cleanest way to make a per-op flag sticky is to bind it via closure — `is_differentiable` lives in the `check` function's enclosing scope and is unreachable from outside. That's the same mechanism `wrap_forward_fn(fn, is_differentiable=False)` uses to make the per-op flag persist across all calls to `eq` or `argmax`.

**Why three gates, all ANDed.** Each is a NECESSARY condition for building a Recipe:
- Gate 1 (global): user disabled grad tracking → no graph.
- Gate 2 (per-op): op is fundamentally non-differentiable → no graph through this node.
- Gate 3 (any input): inputs are all constants → output is constant → no graph needed.

If ANY one fails, the output is a leaf with no Recipe, and backprop simply stops at it.

**Why `globals()` not a bare name.** If `check` is defined inside the factory, a bare `grad_tracking_enabled` reference would first look in the factory's locals (not found), then the module globals — but the binding is captured by NAME, not by value. Re-assigning the module-level name via `set_grad_tracking` is still seen correctly. The explicit `globals()['...']` makes the intent obvious and works even if the function is moved into a different scope later.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()